In [208]:
from bs4 import BeautifulSoup
import asyncio, sys
from concurrent.futures import ThreadPoolExecutor
from markdownify import markdownify as md
import json
def _crawl(url):
    """Run Playwright in a separate thread to avoid Jupyter event loop conflict."""
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url, wait_until="domcontentloaded", timeout=60000)
        html = page.content()
        browser.close()
    return html

# url = "https://www.kapruka.com/buyonline/signature-men-s-formal-slim-fi/kid/ef_pc_clot0v2248pod01092p"
# url = "https://www.kapruka.com/buyonline/mallika-hemachandra-22kt-gold-/kid/jewellerymh091"
url = "https://www.kapruka.com/buyonline/scarlet-petal-vanilla-sponge-g/kid/cake00ka002112"

with ThreadPoolExecutor(max_workers=1) as executor:
    html = executor.submit(_crawl, url).result()

soup = BeautifulSoup(html, "html.parser")
print(soup.title)


<title>Online Scarlet Petal Vanilla Sponge G Online Price in Sri Lanka | Kapruka Cakes Cake</title>


In [209]:
details_div = soup.find("div", id='Tab1')

In [210]:
title = soup.title.string if soup.title else url.split("/")[-1]

test = details_div.find_all(["p", "ul"])
content = ""
for i in test:
    if "<p>" in str(i):
        p_content = i.get_text()
        content += p_content + "\n\n"
    elif "<ul>" in str(i):
        features = "\n".join([li.get_text(strip=True) for li in details_div.find_all('li')])
        content += features + "\n\n"



headings = soup.find_all(["h1", "h2", "h3", "h4"])


In [211]:
print(content)

Looking for a cake delivered fast? Kapruka offers fresh cakes with guaranteed delivery in Sri Lanka.A luscious vanilla sponge soaked in gateaux syrup, layered with smooth whipping cream, and topped with a delicate lily-shaped white chocolate flower. Finished with a radiant red glaze and accented with shimmering gold paper, this cake combines elegance and indulgence.

Ingredient : Vanilla sponge, Gateaux syrup, Whipping cream, White chocolate, Red glaze, Gold paper




In [213]:
import re
main_content = (
            # soup.find('div', {'class': 'tabArea'}) or
            soup.find("div", {"class" : "deliverytextArea"})
        )

In [219]:
main_content

<div class="deliverytextArea">
<div class="stickyBlockDelivery">
<form action="/shops/checkout/deliveryCartViewPage.jsp" id="form1" method="post" name="form1" style="margin:0px;">
<!--/product-information-->
<div class="blockDelivery">
</div>
<div class="blockDelivery mobileRemove">
</div>
<div class="blockDelivery"><span class="mediumbold">Weight: 2.11 Lbs (1.0 KG)<p></p><hr/><p></p></span></div>
<div class="blockDelivery imgtags">
<h1>
				Scarlet Petal Vanilla Sponge Glaze New Year Cake
				
				
				
				
			</h1>
</div>
<div class="product-infoRebrand blockDelivery">
<div class="info-item">
<span class="info-label">Cake Base</span>
<span class="info-value">Vanilla Sponge Base</span>
</div>
</div>
<div class="blockDelivery" style="max-width: 400px">
<a class="addGreeting" href="" style="font-size: 16px;"><span class="glyphicon glyphicon-edit"></span> Add icing greeting (Rs 140)</a>
<div id="cakegreetingP" style="display: none;">
<div class="cgreet-container-unique123" id="cakegreet

In [220]:
content_md = md(str(main_content), heading_style="ATX")

In [222]:
print(content_md)

Weight: 2.11 Lbs (1.0 KG)

---

# Scarlet Petal Vanilla Sponge Glaze New Year Cake

Cake Base
Vanilla Sponge Base

Add icing greeting (Rs 140)

Add

X

You Might Also Need These...

![product_baloonX00171](https://www.kapruka.com/cdn-cgi/image/width=330,quality=93,f=auto/shops/specialGifts/productImages/1662104664046_dsc_2636_m.jpg) See More![](https://www.kapruka.com/shops/customerAccounts/images/arrow_re.png)

![product_baloonX00169](https://www.kapruka.com/cdn-cgi/image/width=330,quality=93,f=auto/shops/specialGifts/productImages/1662104661361_dsc_2594_m.jpg) See More![](https://www.kapruka.com/shops/customerAccounts/images/arrow_re.png)

![product_baloonX00165](https://www.kapruka.com/cdn-cgi/image/width=330,quality=93,f=auto/shops/specialGifts/productImages/1662043397393_dsc_2592_m.jpg) See More![](https://www.kapruka.com/shops/customerAccounts/images/arrow_re.png)

![product_baloonX00164](https://www.kapruka.com/cdn-cgi/image/width=330,quality=93,f=auto/shops/specialGifts/product

In [268]:
def extract_meta(soup, prop):
    """ Return content of a <meta property="..."> tag. """
    tag = soup.find("meta", property=prop)
    return tag['content'].strip() if tag and tag.get("content") else ""

def extract_sku(soup):
    el = soup.find("input",{"id": "id", "type": "hidden"})
    if el and el.get("value"):
        return el["value"]
    return extract_meta(soup, "product:retailer_item_id")

def extract_varients(soup: BeautifulSoup):
    script_tags = soup.find_all('script')
    products_data = {}
    for script in script_tags:
        if script.string and 'let products' in script.string:
            match = re.search(r'let products\s*=\s*(\[.*?\]);', script.string, re.DOTALL)
            if match:
                products_data = json.loads(match.group(1))
                break 
        else:
            availability = extract_meta(soup,"product:availability")
            price_lkr = extract_meta(soup, "product:price:amount")
            product_sku = extract_sku(soup)
            toppers = {}
            text = script.string or ""
            if "allProducts" in text:
                match_toppers = re.search(
                    r'var\s+allProducts\s*=\s*(\[.*?\]);',
                    text
                )
                if match_toppers:
                    toppers = json.loads(match_toppers.group(1))
                    print(toppers)


            price_valid_until = ""
            for script in soup.find_all("script", type="application/ld+json"):
                try:
                    data = json.loads(script.string or "")
                    if str(data.get("@type", "")).lower() in ("product",):
                        offers = data.get("offers", [{}])
                        if isinstance(offers, list):
                            offers = offers[0]
                        price_lkr         = price_lkr or str(offers.get("price", ""))
                        price_valid_until  = offers.get("priceValidUntil", "")
                        avail_url          = offers.get("availability", "")
                        available          = "InStock" in avail_url or available
                except Exception:
                    pass
            products_data = {
                "availabilty" :"in stock" if availability else "Out of stock",
                "price_lkr" : price_lkr, 
                "product sku" : product_sku,
                "price valid until" : price_valid_until
            }

    return products_data

In [269]:
products_details = extract_varients(soup)

[{'productID': 'baloonX00171', 'name': 'Baby Shark Cartoon Theme Foil Balloon Set, 16 Pcs Set For Birthday Decoration Blue', 'account_id': 'Dubaiship00091', 'tags': '<RUSH><FEED_UBER><PARTY_ACCESSORIES>', 'orderLimit': '1', 'vendor': 'Imported By Kapruka', 'available': 'yes', 'price': '9.92', 'price_lk': '17', 'price_1': 'NA', 'price_market': 0.0, 'priceOriginal': '9.92', 'deliveryType': 'Standard', 'description': '<!-- AIKAPRUKA -->\n<p>Make your party fun and colorful with the Baby Shark Cartoon Theme Foil Balloon Set. Perfect for birthdays and baby showers, this set includes 16 beautiful balloons to brighten any celebration.</p>\n\n<ul>\n<li><strong>Pack of 16 Balloons</strong>: Plenty to decorate your space.</li>\n<li><strong>Aluminum Film Design</strong>: Durable and shiny for a festive look.</li>\n<li><strong>Helium or Air Inflation</strong>: Easy to fill with helium or air.</li>\n<li><strong>Fun Theme</strong>: Features Baby Shark characters that kids love.</li>\n</ul>\n\n<p>Ide

In [259]:
products_details

{'availabilty': 'in stock',
 'price_lkr': '4800',
 'product sku': 'cake00ka002112',
 'price valid until': '2026-04-27'}